# Classification d'images à l'aide d'algorithmes de Deep Learning

Projet n&#8239;$^\text{o}$ 6 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 23/03/2026

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

## Prérequis : installer les pilotes GPU, etc.

1. Installer ou mettre à jour le pilote pour la GPU. Par ici pour les GPU Nvidia : https://www.nvidia.com/en-us/drivers/ (spécifier le bon modèle dans le formulaire)

2. Installer le dernier CUDA toolkit (v13.2) de chez Nvidia : 

   - soit de la façon [`runfile (local)`][2] :

     ```bash
     # télécharger l'archive (taille approximative : 4,1 Go)
     wget https://developer.download.nvidia.com/compute/cuda/13.2.0/local_installers/cuda_13.2.0_595.45.04_linux.run

     # exécuter l'archive
     sudo sh cuda_13.2.0_595.45.04_linux.run
     ```

   - soit de la façon [`deb (network)`][3] :

     ```bash
     wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb
     
     sudo dpkg -i cuda-keyring_1.1-1_all.deb
     
     sudo apt update
     
     sudo apt -y install cuda-toolkit-13-2
     ```

3. Ajouter les chemins suivants aux variables d'environnement PATH et LD_LIBRARY_PATH :

     ```bash
     echo 'export PATH=/usr/local/cuda/bin:$PATH' >> ~/.bashrc
     echo 'export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH' >> ~/.bashrc

     # Enfin, recharger .bashrc
     source ~/.bashrc
     ```

4. Installer le backend pour Keras (PyTorch, TensorFlow ou JAX) :

   - Pour PyTorch : Suivre [les instructions de la doc PyTorch][1] : Installer le backend `torch` and `torchvision` compatibles CUDA `>=13.0` pour les GPU Nvidia (ou ROCm `>=7.2` pour les GPU AMD).

   - Si besoin, spécifier dans `pyproject.toml` des bornes plus strictes à la version de Python : par exemple, non pas `>=3.12` mais `>=3.12,<3.14`.

5. Installer Keras : Suivre [les instructions de la doc Keras][4]

[1]: https://pytorch.org/get-started/locally/
[2]: https://developer.nvidia.com/cuda-downloads?target_os=Linux&target_arch=x86_64&Distribution=Ubuntu&target_version=24.04&target_type=runfile_local
[3]: https://developer.nvidia.com/cuda-downloads?target_os=Linux&target_arch=x86_64&Distribution=Ubuntu&target_version=24.04&target_type=deb_network
[4]: https://keras.io/getting_started/

# Notebook de création et d’entraînement du modèle personnel

## Imports et fonctions

In [ ]:
from functions_img_preprocessing import (
    apply_gaussian_blur,
    convert_to_grayscale,
    crop_image,
    equalize_histogram,
    get_boundingbox,
    get_breed,
    mirror_image,
    normalize_image,
    resize_image,
    whiten_image,
)
import os
os.environ["KERAS_BACKEND"] = "torch" # valoriser cette variable d'environnement avant d'importer keras
from collections import  Counter
from keras import layers
from keras.utils import image_dataset_from_directory
from lxml import etree
from pathlib import Path
from PIL import Image
import cv2
import glob
import keras
import numpy as np
import pandas as pd
import scipy.io
import torch


## Étapes préliminaires de test de l'installation

### Tester l'installation de CUDA toolkit et de PyTorch

In [2]:
display(
    f"PyTorch version: {torch.__version__}",
    f"Is CUDA available: {torch.cuda.is_available()}",
    f"CUDA device count: {torch.cuda.device_count()}",
    f"Current device: {torch.cuda.current_device()}",
    f"Device name: {torch.cuda.get_device_name(0)}",
)

'PyTorch version: 2.11.0+cu130'

'Is CUDA available: True'

'CUDA device count: 1'

'Current device: 0'

'Device name: NVIDIA GeForce RTX 2070 with Max-Q Design'

### Tester l'installation de Keras

In [3]:
help(keras)

Help on package keras:

NAME
    keras - DO NOT EDIT.

DESCRIPTION
    This file was autogenerated. Do not edit it by hand,
    since your modifications would be overwritten.

PACKAGE CONTENTS
    _tf_keras (package)
    activations (package)
    applications (package)
    backend (package)
    callbacks (package)
    config (package)
    constraints (package)
    datasets (package)
    distillation (package)
    distribution (package)
    dtype_policies (package)
    export (package)
    initializers (package)
    layers (package)
    legacy (package)
    losses (package)
    metrics (package)
    mixed_precision (package)
    models (package)
    ops (package)
    optimizers (package)
    preprocessing (package)
    quantizers (package)
    random (package)
    regularizers (package)
    saving (package)
    src (package)
    tree (package)
    utils (package)
    visualization (package)
    wrappers (package)

VERSION
    3.14.0

FILE
    /home/mira/kiril_dev/oc-dogs-cv-dl/.venv/lib

## Entraînement du modèle de prédiction *from scratch*

Recenser tous les répertoires contenant les images et les annotations :

In [6]:
img_file_paths = tuple(Path('images/').glob('*/'))
annot_file_paths = tuple(Path('annotations/').glob('*/'))

img_file_paths[0]

PosixPath('images/n02105251-briard')

Les répertoires contenant les images, triés en fonction du nombre d'images :

In [7]:
bag_of_file_counts = {
    # dir_path.as_posix().split('/')[1]: sum(1 for f in dir_path.iterdir() if f.is_file())
    dir_path: sum(1 for f in dir_path.iterdir() if f.is_file())
    for dir_path in img_file_paths
}
bag_of_file_counts_sorted_desc = dict(sorted(bag_of_file_counts.items(), reverse=True, key=lambda item: item[1]))
display(
    "Les répertoires contenant le plus d'images :",
    dict(list(bag_of_file_counts_sorted_desc.items())[:5]),
    "Les répertoires contenant le moins d'images :",
    dict(list(bag_of_file_counts_sorted_desc.items())[-5:]),
)

"Les répertoires contenant le plus d'images :"

{PosixPath('images/n02085936-Maltese_dog'): 252,
 PosixPath('images/n02088094-Afghan_hound'): 239,
 PosixPath('images/n02092002-Scottish_deerhound'): 232,
 PosixPath('images/n02112018-Pomeranian'): 219,
 PosixPath('images/n02090721-Irish_wolfhound'): 218}

"Les répertoires contenant le moins d'images :"

{PosixPath('images/n02115913-dhole'): 150,
 PosixPath('images/n02106166-Border_collie'): 150,
 PosixPath('images/n02101556-clumber'): 150,
 PosixPath('images/n02086079-Pekinese'): 149,
 PosixPath('images/n02090379-redbone'): 148}

### Expérience 1 : Premier entraînement à 3 classes, d'abord sans *data augmentation*, ensuite avec

Nous allons donc sélectionner trois races selon le nombre d'images mises à notre disposition :

- Deux races contenant le plus d'images : **`Maltese_dog`** (252 images) et **`Afghan_hound`** (239 images).
- Une race contenant le moins d'images : **`redbone`** (148 images).

In [8]:
paths_to_train_input_img = tuple(list(bag_of_file_counts_sorted_desc.keys())[i] for i in [-1, 0, 1])
paths_to_train_input_img

(PosixPath('images/n02090379-redbone'),
 PosixPath('images/n02085936-Maltese_dog'),
 PosixPath('images/n02088094-Afghan_hound'))

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 64
EPOCHS     = 30
LR         = 1e-3
NUM_CLASSES = 120

data_augm = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomCrop(IMG_SIZE, IMG_SIZE),
    # layers.RandomRotation(0.1),   # ← décommenter selon besoin
    # layers.RandomZoom(0.1),
], name="data_augmentation")

def build_model(n_classes: int = 120, include_data_augm: bool = True):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = data_augm(inputs) if include_data_augm else inputs
    x = layers.Rescaling(1./255)(x)

    # Bloc 1 ─ CHANGER : filtres, taille kernel, ajouter BatchNorm
    x = layers.Conv2D(32, 5, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

    # Bloc 2 ─ CHANGER : ajouter d'autres blocs conv ici
    x = layers.Conv2D(64, 5, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(512, activation="relu")(x)
    # x = layers.Dropout(0.5)(x)   # ← décommenter pour régulariser

    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

E0000 00:00:1776882225.359297 1149167 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776882225.366518 1149167 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


NameError: name 'IMG_SIZE' is not defined

#### a) Entraînement à 3 classes sans *data augmentation*

In [ ]:
...

Ellipsis

#### b) Entraînement à 3 classes avec *data augmentation*

In [ ]:
...

Ellipsis